# López González Pamela Citlali Atletl
* scara_tray_line_py.py:
    #!/usr/bin/env python3

        #  ==========================================
        # IMPORTACIÓN DE LIBRERÍAS
        # ==========================================
        import rclpy
        from rclpy.node import Node
        # Mensajes para controlar las trayectorias de los motores (articulaciones)
        from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint
        from builtin_interfaces.msg import Duration

        import time
        from math import cos, sin, acos, asin, atan2, sqrt

        # ==========================================
        # CLASE PRINCIPAL DEL NODO
        # ==========================================
        class ScaraTrayLineNode(Node):
            def __init__(self):
                # Inicializa el nodo de ROS 2 con un nombre identificativo
                super().__init__("scara_tray_line_node")
                
                # Tópico oficial del controlador de ROS para mover los motores
                topic_name = "/scara_trajectory_controller/joint_trajectory"
                
                # Nombres de las articulaciones del robot (deben coincidir con el archivo Xacro/URDF)
                self.joints_ = ['link_1_joint', 'link_2_joint', 'link_3_joint']
        
        # Variables de control de tiempo para la trayectoria
        self.lamda_ = 0          # Contador de pasos actuales (va de 0 a 10)
        self.Tiempo_ejec_ = 10   # Tiempo total estimado para terminar la línea recta (10 segundos)

        # Crea el Publicador que enviará los mensajes de movimiento al robot
        self.scara_tray_pub_ = self.create_publisher(JointTrajectory, topic_name, 10)
        
        # Crea un Temporizador que ejecuta la función 'trayectory_cbck' cada 1 segundo
        self.tray_timer_ = self.create_timer(1, self.trayectory_cbck)
        
        # Imprime un mensaje en la terminal avisando que el script arrancó con éxito
        self.get_logger().info('Scara activo, trayectoria linea recta')

        # ------------------------------------------
        # FUNCIÓN TEMPORIZADA (SE EJECUTA CADA SEGUNDO)
        # ------------------------------------------
        def trayectory_cbck(self):
            # Prepara los contenedores vacíos para estructurar el mensaje de ROS 2
            trayectory_msg = JointTrajectory()
            trayectory_msg.joint_names = self.joints_ # Indica a qué motores va la orden
            point = JointTrajectoryPoint()            # Estructura para almacenar los ángulos de llegada

            # FASE 1: Mientras estemos dentro de los 10 segundos de movimiento
            if self.lamda_ <= self.Tiempo_ejec_:
                # Coordenadas espaciales de INICIO (X, Y en metros, Ángulo del efector final)
                x_1 = 0.1
                y_1 = 0.6
                theta_1 = 0
                
                # Coordenadas espaciales de DESTINO FINAL
                x_2 = 0.3
                y_2 = -0.6
                theta_2 = 1.57 # (Equivale a 90 grados en radianes)
            
            # CINEMÁTICA INVERSA: Calcula qué ángulos de motores se necesitan para este paso de la línea
            solucion = invk_sol(self.lamda_, x_1, y_1, theta_1, x_2, y_2, theta_2)
            
            # Asigna los ángulos calculados al punto de la trayectoria
            point.positions = solucion
            point.time_from_start = Duration(sec=1) # Le pide al robot alcanzar la posición en 1 segundo
            
            # Empaqueta y publica el comando al simulador Gazebo
            trayectory_msg.points.append(point)
            self.scara_tray_pub_.publish(trayectory_msg)
            
            # Muestra en la terminal los ángulos calculados en tiempo real
            self.get_logger().info("Postura actual {}".format(solucion))
            
            # Pausa el script 2 segundos para dar estabilidad y avanza al siguiente paso de la línea
            time.sleep(2)
            self.lamda_ += 1
            
        # FASE 2: Cuando ya se completó el recorrido (Mayor a 10 segundos)
        elif self.lamda_ > 10:
            # Resetea de forma segura los valores de las articulaciones a cero
            link_1_joint = 0
            link_2_joint = 0
            link_3_joint = 0
            return [float(link_1_joint), float(link_2_joint), float(link_3_joint)]

        # ==========================================
        # FUNCIÓN DE MATEMÁTICAS: CINEMÁTICA INVERSA
        # ==========================================
def invk_sol(param, x_in, y_in, theta_in, x_fin, y_fin, theta_fin):
    Tiempo_ejec_ = 10
    
    # Dimensiones físicas reales de los eslabones del robot (coinciden con el diseño)
    L_1 = 0.5 # 50 cm
    L_2 = 0.5 # 50 cm
    L_3 = 0.3 # 30 cm
    
    # INTERPOLACIÓN LINEAL: Calcula el punto exacto en la línea recta en el que debe estar el robot
    # basándose en el segundo actual ('param').
    x_P = x_in + (param / Tiempo_ejec_) * (x_fin - x_in)
    y_P = y_in + (param / Tiempo_ejec_) * (y_fin - y_in)
    theta_P = theta_in + (param / Tiempo_ejec_) * (theta_fin - theta_in)
    
    # Resta el efecto del tercer eslabón (L3) para encontrar la posición de la muñeca del robot (X3, Y3)
    x_3 = x_P - L_3 * cos(theta_P)
    y_3 = y_P - L_3 * sin(theta_P)
    
    # LEY DE COSENOS: Resuelve trigonométricamente los ángulos de los motores 1, 2 y 3
    # Ángulo del motor 2 (Articulación del codo)
    theta_2 = acos((pow(x_3, 2) + pow(y_3, 2) - pow(L_1, 2) - pow(L_2, 2)) / (2 * L_1 * L_2))
    
    # Ángulo del motor 1 (Articulación del hombro/base)
    beta = atan2(y_3, x_3)
    psi = acos((pow(x_3, 2) + pow(y_3, 2) + pow(L_1, 2) - pow(L_2, 2)) / (2 * L_1 * sqrt(pow(x_3, 2) + pow(y_3, 2))))
    theta_1 = beta - psi
    
    # Ángulo del motor 3 (Orientación de la herramienta/punta)
    theta_3 = theta_P - theta_1 - theta_2
    
    # Devuelve los tres ángulos listos para ser enviados a los motores
    return [float(theta_1), float(theta_2), float(theta_3)]

    # ==========================================
    # FUNCIÓN DE ARRANQUE (MAIN)
    # ==========================================
        def main(args=None):
            rclpy.init(args=args)       # Inicia la comunicación general de ROS 2
            node = ScaraTrayLineNode()  # Instancia el nodo que creamos arriba
            rclpy.spin(node)            # Mantiene el script vivo escuchando y publicando continuamente
            rclpy.shutdown()            # Cierra el nodo limpiamente al terminar

        if __name__ == "__main__":
            main()

* dofbot_sequence_py.py
        #!/usr/bin/env python3

        # =============================================================================
        # IMPORTACIÓN DE LIBRERÍAS
        # =============================================================================
        import rclpy
        from rclpy.node import Node
        # Mensajes estándar de ROS 2 para controlar trayectorias de articulaciones
        from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint
        from builtin_interfaces.msg import Duration

        import time
        from math import cos, sin, acos, asin, atan2, sqrt

        # =============================================================================
        # CLASE PRINCIPAL DEL CONTROLADOR DEL ROBOT DOFBOT
        # =============================================================================
        class DofbotControlNode(Node):
            def __init__(self):
                # Inicializa el nodo de ROS 2
                super().__init__("dofbot_tray_control_node")
                
                # self.lamda_ actúa como un contador de pasos (Máquina de Estados)
                self.lamda_ = 0
                
                # Tópicos de ROS 2 para enviar comandos al brazo y a la pinza (gripper)
                topic_dofbot_ = "/dofbot_trajectory_controller/joint_trajectory"
                topic_gripper_ = "/dofbot_gripper_controller/joint_trajectory"
                
                # Configuración del Publicador y nombres de los 5 motores del brazo
                self.dofbot_publisher_ = self.create_publisher(JointTrajectory, topic_dofbot_, 10)
                self.dofbot_joints_ = ['arm_joint_01', 'arm_joint_02', 'arm_joint_03', 'arm_joint_04', 'arm_joint_05']
                
                # Configuración del Publicador y nombres de las articulaciones de la pinza
                self.gripper_publisher_ = self.create_publisher(JointTrajectory, topic_gripper_, 10)
                self.gripper_joints_ = ['grip_joint', 'rfinger_joint_01', 'rfinger_joint_02', 
                                        'lfinger_grip_joint_01', 'lfinger_grip_joint_02', 'lfinger_grip_joint_03']
                
                # Temporizador: Ejecuta la función 'timer_callback' cada 0.5 segundos
                self.timer_ = self.create_timer(0.5, self.timer_callback)
                self.get_logger().info('Nodo de control del dofbot en funcionamiento')

            # -------------------------------------------------------------------------
            # SECUENCIA DE MOVIMIENTOS (MÁQUINA DE ESTADOS)
            # -------------------------------------------------------------------------
            def timer_callback(self):
                # Prepara las estructuras de mensajes vacías para el brazo y la pinza
                dofbot_msg = JointTrajectory()
                dofbot_msg.joint_names = self.dofbot_joints_
                dofbot_point = JointTrajectoryPoint()

                gripper_msg = JointTrajectory()
                gripper_msg.joint_names = self.gripper_joints_
                gripper_point = JointTrajectoryPoint()

                # =====================================================================
                # PASOS 0 AL 2: PRUEBAS INICIALES DE LA PINZA (ABRIR / CERRAR)
                # =====================================================================
                if self.lamda_ == 0:
                    # PASO 0: Abrir pinza por primera vez (1.57 radianes)
                    gstate = 1.57
                    gripper_st = gripper_state(gstate)
                    gripper_point.positions = gripper_st
                    gripper_point._time_from_start = Duration(sec=1)
                    gripper_msg.points.append(gripper_point)
                    self.gripper_publisher_.publish(gripper_msg)
                    self.get_logger().info('Gripper open')
                    time.sleep(5) # Espera 5 segundos antes de avanzar
                    self.lamda_ += 1

                elif self.lamda_ == 1:
                    # PASO 1: Cerrar pinza (0 radianes)
                    gstate_2 = 0
                    gripper_st = gripper_state(gstate_2)
                    gripper_point.positions = gripper_st
                    gripper_point._time_from_start = Duration(sec=1)
                    gripper_msg.points.append(gripper_point)
                    self.gripper_publisher_.publish(gripper_msg)
                    self.get_logger().info('Gripper close')
                    time.sleep(5)
                    self.lamda_ += 1

                elif self.lamda_ == 2:
                    # PASO 2: Volver a abrir la pinza para dejarla lista para sujetar un objeto
                    gstate = 1.57
                    gripper_st = gripper_state(gstate)
                    gripper_point.positions = gripper_st
                    gripper_point._time_from_start = Duration(sec=1)
                    gripper_msg.points.append(gripper_point)
                    self.gripper_publisher_.publish(gripper_msg)
                    self.get_logger().info('Gripper open')
                    time.sleep(5)
                    self.lamda_ += 1

                # =====================================================================
                # PASOS 3 AL 7: SECUENCIA DE RECOGIDA Y TRANSPORTE (PICK AND PLACE)
                # =====================================================================
                elif self.lamda_ == 3:
                    # PASO 3: Mover el brazo a la PRIMERA POSTURA (Aproximación al objeto)
                    x_1, y_1, z_1 = 0.2, 0.0, 0.05
                    theta_p_1 = 3.1416 * (3/4) # Ángulo de inclinación de la muñeca
                    theta_g_1 = 0.0            # Rotación de la punta
                    
                    # Calcula los ángulos de los motores usando Cinemática Inversa
                    solution_pos = dofbot_ink(x_1, y_1, z_1, theta_p_1, theta_g_1)
                    dofbot_point.positions = solution_pos
                    dofbot_point.time_from_start = Duration(sec=2)
                    dofbot_msg.points.append(dofbot_point)
                    self.dofbot_publisher_.publish(dofbot_msg)
                    self.get_logger().info('poture {}'.format(solution_pos))
                    time.sleep(15) # Da 15 segundos para asegurar que el brazo físico llegue
                    self.lamda_ += 1

                elif self.lamda_ == 4:
                    # PASO 4: Cerrar pinza para sujetar/tomar el objeto
                    gstate_2 = 0
                    gripper_st = gripper_state(gstate_2)
                    gripper_point.positions = gripper_st
                    gripper_point._time_from_start = Duration(sec=2)
                    gripper_msg.points.append(gripper_point)
                    self.gripper_publisher_.publish(gripper_msg)
                    self.get_logger().info('Gripper close')
                    time.sleep(10)
                    self.lamda_ += 1

                elif self.lamda_ == 5:
                    # PASO 5: SEGUNDA POSTURA (Levantar el objeto verticalmente a Z=0.11m)
                    x_2, y_2, z_2 = 0.15, 0.0, 0.11
                    theta_p_2 = 3.1416 * (3/4)
                    theta_g_2 = 0 
                    solution_pos = dofbot_ink(x_2, y_2, z_2, theta_p_2, theta_g_2)
                    dofbot_point.positions = solution_pos
                    dofbot_point.time_from_start = Duration(sec=2)
                    dofbot_msg.points.append(dofbot_point)
                    self.dofbot_publisher_.publish(dofbot_msg)
                    time.sleep(15)
                    self.lamda_ += 1

                elif self.lamda_ == 6:
                    # PASO 6: TERCERA POSTURA (Girar el brazo en el espacio hacia un lado: Y=0.15m)
                    x_3, y_3, z_3 = 0.15, 0.15, 0.11
                    theta_p_3 = 3.1416 * (3/4) 
                    theta_g_3 = 0 
                    solution_pos = dofbot_ink(x_3, y_3, z_3, theta_p_3, theta_g_3)
                    dofbot_point.positions = solution_pos
                    dofbot_point.time_from_start = Duration(sec=2)
                    dofbot_msg.points.append(dofbot_point)
                    self.dofbot_publisher_.publish(dofbot_msg)
                    time.sleep(15)
                    self.lamda_ += 1

                elif self.lamda_ == 7:
                    # PASO 7: CUARTA POSTURA (Bajar el brazo para colocar el objeto en el suelo: Z=0.05m)
                    x_3, y_3, z_3 = 0.15, 0.15, 0.05
                    theta_p_3 = 3.1416 * (3/4) 
                    theta_g_3 = 0 
                    solution_pos = dofbot_ink(x_3, y_3, z_3, theta_p_3, theta_g_3)
                    dofbot_point.positions = solution_pos
                    dofbot_point.time_from_start = Duration(sec=2)
                    dofbot_msg.points.append(dofbot_point)
                    self.dofbot_publisher_.publish(dofbot_msg)
                    time.sleep(15)
                    self.lamda_ += 1

                # =====================================================================
                # PASOS 8 AL 9: LIBERACIÓN Y RETORNO A POSICIÓN DE REPOSO
                # =====================================================================
                elif self.lamda_ == 8:
                    # PASO 8: Abrir la pinza para soltar el objeto en su destino
                    gstate = 1.57
                    gripper_st = gripper_state(gstate)
                    gripper_point.positions = gripper_st
                    gripper_point._time_from_start = Duration(sec=2)
                    gripper_msg.points.append(gripper_point)
                    self.gripper_publisher_.publish(gripper_msg)
                    self.get_logger().info('Gripper open')
                    time.sleep(10)
                    self.lamda_ += 1

                elif self.lamda_ == 9:
                    # PASO 9: Regresar todos los motores del brazo a 0 grados (Postura de reposo / "Home")
                    solution_pos = [float(0.0), float(0.0), float(0.0), float(0.0), float(0.0)]
                    dofbot_point.positions = solution_pos
                    dofbot_point.time_from_start = Duration(sec=2)
                    dofbot_msg.points.append(dofbot_point)
                    self.dofbot_publisher_.publish(dofbot_msg)
                    time.sleep(10)
                    # Nota: Al no aumentar self.lamda_, el robot se quedará en esta postura indefinidamente.

        # =============================================================================
        # FUNCIÓN MATEMÁTICA: CINEMÁTICA INVERSA DEL ROBOT DOFBOT (5 GRADOS DE LIBERTAD)
        # =============================================================================
        def dofbot_ink(x_P, y_P, z_P, theta_1_P, theta_g):
            # Dimensiones físicas del robot Dofbot (longitud de eslabones en metros)
            z_0_1 = 0.105 # Altura desde el suelo a la primera articulación
            L_1 = 0.084   # Eslabón del hombro al codo
            L_2 = 0.084   # Eslabón del codo a la muñeca
            L_3 = 0.115   # Longitud de la pinza/herramienta

            # Cálculo trigonométrico clásico para resolver la cinemática inversa en 3D:
            theta_1 = atan2(y_P, x_P) # Orientación de la base (Motor 1)
            
            # Variables auxiliares que restan el efecto geométrico del eslabón de la pinza (L3)
            aux_x = sqrt(pow(x_P, 2) + pow(y_P, 2)) - L_3 * sin(theta_1_P)
            aux_z = z_P - z_0_1 - L_3 * cos(theta_1_P)
            
            # Aplicación de la Ley de Cosenos para calcular las articulaciones intermedias
            norm_4_P = sqrt(pow(aux_z, 2) + pow(aux_x, 2))
            epsilon = acos(aux_z / norm_4_P)
            alpha = acos((pow(L_1, 2) + pow(norm_4_P, 2) - pow(L_2, 2)) / (2 * L_1 * norm_4_P))
            
            theta_2 = epsilon - alpha  # Ángulo del Hombro (Motor 2)
            theta_3 = 3.1416 - asin((sin(alpha) * sqrt(pow(aux_x, 2) + pow(aux_z, 2))) / (L_2)) # Ángulo del Codo (Motor 3)
            theta_4 = theta_1_P - theta_2 - theta_3 # Ángulo de inclinación de la Muñeca (Motor 4)
            theta_5 = theta_g          # Ángulo de rotación de la punta (Motor 5)
            
            # Devuelve los 5 ángulos ordenados con los signos cinemáticos corregidos para los actuadores
            return [float(theta_1), float(-theta_2), float(-theta_3), float(-theta_4), float(theta_5)]

        # -----------------------------------------------------------------------------
        # FUNCIÓN AUXILIAR: REPARTO DE ÁNGULOS PARA LOS ESLABONES DE LA PINZA
        # -----------------------------------------------------------------------------
        def gripper_state(theta):
            # Dado que las pinzas paralelas tienen engranajes acoplados mecánicamente,
            # cuando un dedo se mueve de forma positiva, el otro debe moverse de forma negativa para cerrar simétricamente.
            return [float(-theta), float(theta), float(-theta), float(theta), float(-theta), float(theta)]

        # =============================================================================
        # ARRANQUE DE ROS 2 (MAIN)
        # =============================================================================
        def main(args=None):
            rclpy.init(args=args)       # Inicia el entorno de comunicación de ROS 2
            node = DofbotControlNode()  # Instancia nuestra clase controladora
            rclpy.spin(node)            # Mantiene el nodo activo corriendo en bucle
            rclpy.shutdown()            # Cierra el proceso de forma limpia al salir

        if __name__ == "__main__":
            main()